In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

In [3]:
Cybersecurity_agent = Agent(
    name="Cybersecurity Assistant",
    instructions="""
    You are a cybersecurity awareness assistant.
    Explain cybersecurity concepts in simple language.
    Help users identify phishing emails and online scams.
    Suggest ways to protect accounts and personal information.
    Give short, clear, and practical answers.
    """,
)

In [4]:
with trace("Cybersecurity Agent"):
    result = await Runner.run(
        Cybersecurity_agent,
        "How can I identify a phishing email?"
    )

print(result)

RunResult:
- Last agent: Agent(name="Cybersecurity Assistant", ...)
- Final output (str):
    Here’s a quick, practical checklist to spot phishing emails:
    
    - Sender details: unfamiliar or mismatched email address (look at the domain, not just the display name).
    - Urgency or fear: “Act now”, “Hurry”, or threats to lock your account.
    - Spelling/grammar: many typos or odd phrasing.
    - Suspicious links: don’t click. Hover over links to see the real URL; if it doesn’t match the company, don’t click.
    - Unusual requests: asks for passwords, payment info, or personal data.
    - Unexpected attachments: especially PDFs or ZIP files from unknown senders.
    - Generic greeting: “Dear customer” instead of your name.
    - Brand inconsistencies: inconsistent logos, formatting, or email signature.
    - Email context: irrelevant to you or the company you deal with.
    
    What to do if you’re unsure:
    - Don’t click or open attachments.
    - Verify via a trusted channel:

In [5]:
response_stream = Runner.run_streamed(
    Cybersecurity_agent,
    "How can I protect myself from online scams?"
)

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Here are quick, practical steps to protect yourself from online scams:

- Use strong, unique passwords for every account; use a password manager.
- Turn on two-factor authentication (MFA) wherever possible.
- Keep devices and apps updated with the latest security patches.
- Be wary of unexpected messages (email, text, social) asking for personal info or pesos. Don’t click links or open attachments from unknown senders.
- Check URLs carefully. Hover links to preview the site, and if in doubt, type the official site address directly in your browser.
- Don’t share codes or passwords, even if someone claims to be from a trusted company or “tech support.”
- Verify requests via a separate channel (call the company or use the official app/site) if someone asks for sensitive info.
- Don’t pay or offer info for “verification,” “refunds,” or “prizes.”
- Use reputable security software and enable automatic scans.
- Regularly back up important data (and keep backups offline or in a secure cloud).
